# Session 3 – Building Rule-Based AI: Student Tasks

**Module:** CPU5006-20 Artificial Intelligence
**Session:** 3 of 12
**Student name:** ____________________
**Date:** ____________________

## How to use this notebook
This notebook goes with the **Session 3: Building Rule-Based AI** slides. Each task here matches a **Task N** on the slides, so work through the notebook alongside the session.

- The **warm-up** cells are the code shown on the slides. Run them first.
- Write your answers for **Task 1** to **Task 4** in the code cells provided.
- Run each cell and check the output.
- Work with a partner where the slides say **"in pairs"**.

### Learning outcomes
By the end of this session, you should be able to:
- implement **forward chaining** and **backward chaining** in Python
- turn a dataset row into **facts** and write **rules** over it
- **evaluate** a rule-based classifier against a baseline
- start structuring your **S1** project

## Warm-up: the code from the slides

Run these cells first. They are the same `facts`, `rules`, `forward_chain`, `backward_chain` and `predict` code shown in **Chaining in Code** and **Backward Chaining in Code**. You will reuse this pattern in Tasks 1 to 3.

In [1]:
facts = {"has_fur", "gives_milk", "eats_meat", "has_stripes"}

rules = [
    ({"has_fur", "gives_milk"}, "mammal"),
    ({"has_feathers", "lays_eggs"}, "bird"),
    ({"mammal", "eats_meat"}, "carnivore"),
    ({"carnivore", "has_stripes"}, "tiger"),
    ({"mammal", "has_hooves"}, "ungulate"),
    ({"ungulate", "has_stripes"}, "zebra"),
]

### Forward chaining

`forward_chain(facts, rules)` keeps applying rules until nothing new can be derived.

In [19]:
def forward_chain(facts, rules):
    # a set condenses a table of {1, 2, 2, 3, 5, 3, 6} to {1, 2, 3, 5, 6}
    known = set(facts)              # never change the caller's facts
    changed = True
    while changed:                  # repeat until nothing new is derived
        changed = False
        for conditions, conclusion in rules: # ({"has_fur", "gives_milk"}, <- conditions, rules -> "mammal"),
            # print(len(conditions), len(known))
            # print(f"conditions: {conditions}")
            # print(f"known: {known}")
            # checks if conditions is a subset of known (WHAT?!)
            if conditions <= known and conclusion not in known:
                # print(known, conclusion)
                known.add(conclusion)
                changed = True
    return known

print(sorted(forward_chain(facts, rules)))

['carnivore', 'eats_meat', 'gives_milk', 'has_fur', 'has_stripes', 'mammal', 'tiger']


### Forward chaining with an explanation

This version also records **why** each fact was derived.

In [18]:
def forward_chain_trace(facts, rules):
    known, trace = set(facts), []
    changed = True
    while changed:
        changed = False
        for conditions, conclusion in rules:
            if conditions <= known and conclusion not in known:
                known.add(conclusion)
                trace.append((sorted(conditions), conclusion))   # remember WHY
                changed = True
    return known, trace

known, trace = forward_chain_trace(facts, rules)
for conditions, conclusion in trace:
    print(f"{' + '.join(conditions)}  ->  {conclusion}")

gives_milk + has_fur  ->  mammal
eats_meat + mammal  ->  carnivore
carnivore + has_stripes  ->  tiger


### Backward chaining

`backward_chain(goal, facts, rules)` starts from a **goal** and works backwards, proving each sub-goal in turn.

In [20]:
def backward_chain(goal, facts, rules):
    if goal in facts:                        # already known: proved
        return True
    for conditions, conclusion in rules:
        if conclusion == goal:               # a rule that could prove the goal
            if all(backward_chain(c, facts, rules) for c in sorted(conditions)):
                return True
    return False

print(backward_chain("tiger", facts, rules))    # True
print(backward_chain("zebra", facts, rules))    # False

True
False


### From a dataset row to a prediction

Turn a dataset row into `attribute=value` facts, run forward chaining, then decide between rules that fire using **first match wins**.

In [21]:
row = {"buying": "low", "safety": "high", "persons": 4}

# every attribute=value pair becomes a fact
row_facts = {f"{attribute}={value}" for attribute, value in row.items()}
print(sorted(row_facts))

car_rules = [
    ({"safety=low"}, "class=unacc"),
    ({"persons=2"}, "class=unacc"),
    ({"safety=high", "buying=low"}, "class=good"),
    ({"safety=high", "buying=med"}, "class=good"),
    ({"persons=4"}, "class=acc"),
]

def predict(row, rules, default="class=unacc"):
    """First rule in the list that fires wins (conflict resolution)."""
    facts = {f"{a}={v}" for a, v in row.items()}
    for conditions, conclusion in rules:        # order = priority
        if conditions <= facts:
            return conclusion
    return default

print(predict({"buying": "low", "safety": "high", "persons": 4}, car_rules))
print(predict({"buying": "low", "safety": "low", "persons": 4}, car_rules))

['buying=low', 'persons=4', 'safety=high']
class=good
class=unacc


---
## Task 1 – Forward chaining on California housing  ⏱ 20 min · in pairs

Build a rule-based system using **forward chaining** with the **California housing** dataset:

1. turn numeric features into **categories** (e.g. `income=low`)
2. write at least **five rules** that conclude `price=high` or `price=low`
3. use **first match wins** with a default
4. report your accuracy

### Stretch:
Compare against always predicting the most common class.

In [19]:
%pip install pandas


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
%pip install scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
from sklearn.datasets import fetch_california_housing

df = fetch_california_housing(as_frame=True).frame
df["price"] = ["high" if v > df["MedHouseVal"].median() else "low"
               for v in df["MedHouseVal"]]

# categorise a numeric column into low / medium / high
df["income"] = ["low" if v < 2.5 else "medium" if v < 4.5 else "high"
                for v in df["MedInc"]]

print(df[["MedInc", "income", "MedHouseVal", "price"]].head())
print(df["price"].value_counts())

   MedInc  income  MedHouseVal price
0  8.3252    high        4.526  high
1  8.3014    high        3.585  high
2  7.2574    high        3.521  high
3  5.6431    high        3.413  high
4  3.8462  medium        3.422  high
price
low     10323
high    10317
Name: count, dtype: int64


In [ ]:
# TODO 1: categorise at least one more numeric column (e.g. AveRooms, HouseAge)
# your_column = ...

# TODO 2: write at least 5 rules as (set of conditions, conclusion) pairs,
#         concluding "price=high" or "price=low"
house_rules = [
    # ({"income=high"}, "price=high"),
]

# TODO 3: complete predict() using first-match-wins, like the warm-up predict()
def predict_price(row, rules, default="price=low"):
    ...

# TODO 4: report your accuracy across every row in df
rows = df.to_dict("records")
# accuracy = ...
# print(f"Accuracy: {accuracy:.2f}")

# Stretch: compare against always predicting the most common price

---
## Task 2 – Backward chaining on the iris dataset  ⏱ 20 min · in pairs

Build a rule-based system using **backward chaining** with the **iris** dataset:

1. look at the data: which features separate the species?
2. turn petal and sepal measurements into **categories**
3. write rules that conclude `species=setosa`, `species=versicolor` or `species=virginica`
4. prove each species goal for every row, and report your accuracy

### Stretch:
Print the proof for one row that your rules classify **wrongly**.

In [15]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True).frame
iris["species"] = iris["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

# look at the data: which features separate the species?
print(iris.groupby("species")[["petal length (cm)", "petal width (cm)"]].describe())

           petal length (cm)                                               \
                       count   mean       std  min  25%   50%    75%  max   
species                                                                     
setosa                  50.0  1.462  0.173664  1.0  1.4  1.50  1.575  1.9   
versicolor              50.0  4.260  0.469911  3.0  4.0  4.35  4.600  5.1   
virginica               50.0  5.552  0.551895  4.5  5.1  5.55  5.875  6.9   

           petal width (cm)                                            
                      count   mean       std  min  25%  50%  75%  max  
species                                                                
setosa                 50.0  0.246  0.105386  0.1  0.2  0.2  0.3  0.6  
versicolor             50.0  1.326  0.197753  1.0  1.2  1.3  1.5  1.8  
virginica              50.0  2.026  0.274650  1.4  1.8  2.0  2.3  2.5  


In [ ]:
# TODO 1: turn petal length and petal width into categories, based on what
#         you saw above (e.g. "petal_length=short")
def row_to_facts(row):
    facts = set()
    # facts.add(...)
    return facts

# TODO 2: write rules that conclude "species=setosa", "species=versicolor"
#         or "species=virginica"
iris_rules = [
    # ({"petal_length=short"}, "species=setosa"),
]

# reuse backward_chain from the warm-up

# TODO 3: for every row, prove each species goal and pick the one that is
#         proved, then compare with the real species and report accuracy
correct = 0
for _, row in iris.iterrows():
    row_facts = row_to_facts(row)
    # proven = [g for g in (...) if backward_chain(g, row_facts, iris_rules)]
    # predicted = proven[0] if proven else "species=unknown"
    # if predicted == f"species={row['species']}":
    #     correct += 1

# accuracy = correct / len(iris)
# print(f"Accuracy: {accuracy:.2f}")

# Stretch: print the proof (using a "prove" function like the slides) for
# one row your rules get wrong

---
## Task 3 – Evaluate your rules  ⏱ 15 min

Using **either** of your rule sets from Tasks 1 and 2:

1. work out the **baseline** accuracy (always predict the most common class)
2. split your data into **train** and **test** (70 / 30)
3. report accuracy on **test**, and list three rows your rules get wrong
4. write one sentence on what you would change

### Stretch:
Compute precision and recall for one class.

In [ ]:
from collections import Counter
import random

# TODO: point these at your rows and classify function from Task 1 or Task 2
# rows = ...
# classify = predict_price   # or your iris classifier

# TODO 1: baseline accuracy (always predict the most common class)
# majority = Counter(r["class"] for r in rows).most_common(1)[0][0]
# baseline = sum(r["class"] == majority for r in rows) / len(rows)
# print(f"Baseline: {baseline:.2f}")

# TODO 2: split into train (70%) and test (30%), with a random seed
random.seed(42)
# shuffled = rows[:]
# random.shuffle(shuffled)
# cut = int(0.7 * len(shuffled))
# train, test = shuffled[:cut], shuffled[cut:]

# TODO 3: report accuracy on test, and list 3 rows your rules get wrong
# test_accuracy = ...
# wrong_rows = [r for r in test if classify(r) != r["class"]][:3]
# print(f"Test accuracy: {test_accuracy:.2f}")
# print(wrong_rows)

# TODO 4: one sentence on what you would change
# I would change: ...

# Stretch: precision and recall for one class

### The quick way: scikit-learn

`sklearn.metrics` does the same job as the `baseline` and `test_accuracy` code above, in one line each. It works the same whether your labels came from a rule-based `classify()` function or a trained model — useful for **S1 and S2 alike**.

Run this once **Task 3** above is complete (it needs `test` and `classify`).

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# TODO: score your held-out test rows the quick way
# y_true = [r["class"] for r in test]
# y_pred = [classify(r) for r in test]

# print(f"Accuracy: {accuracy_score(y_true, y_pred):.2f}")
# print(classification_report(y_true, y_pred, zero_division=0))
# print(confusion_matrix(y_true, y_pred, labels=sorted(set(y_true))))

---
## Task 4 – Start your S1 repository  ⏱ 20 min

1. create your repository and add a **README** and `requirements.txt`
2. load your chosen dataset and print its **class distribution**
3. work out the **baseline** accuracy
4. write your **first three rules** and their accuracy
5. **commit and push**

### Stretch:
Record 2 minutes explaining your first rule out loud, as practice for your video.

> **Outside this notebook:** steps 1 and 5 happen in your terminal and on GitHub, not here.
> 1. `git init` (or create the repo on GitHub first, then clone it)
> 2. create `README.md` and `requirements.txt`
> 3. copy your Task 1 to 4 code into `.py` files, using the project structure from the slides
> 4. `git add .`, `git commit -m "First three rules and baseline"`, `git push`

In [ ]:
import pandas as pd

# TODO 1: load YOUR chosen S1 dataset (Car Evaluation, Credit Risk, Adult or Food)
# df = pd.read_csv("your_dataset.csv")
TARGET = "class"   # change to your target column name

# TODO 2: print the class distribution
# print(df[TARGET].value_counts())

# TODO 3: baseline accuracy
# baseline = df[TARGET].value_counts(normalize=True).max()
# print(f"Baseline: {baseline:.2f}")

# TODO 4: your first three rules, and their accuracy
first_three_rules = [
    # ({"...": "..."}, "..."),
]

def classify(row):
    ...

# rows = df.to_dict("records")
# accuracy = sum(classify(r) == r[TARGET] for r in rows) / len(rows)
# print(f"3-rule accuracy: {accuracy:.2f}")

---
## Reflection

Answer these in a markdown cell below:
1. Which was easier to write: forward chaining rules (Task 1) or backward chaining rules (Task 2)? Why?
2. What did Task 3 tell you about your rules: did they beat the baseline by much?
3. Which dataset are you using for S1, and what is your first rule?

## My reflection

*Write here...*